In [1]:
# get text from sample pdfs
from src.basic_code_search.document_loader import load_pdf_documents

text = load_pdf_documents(folder_path="data")
print(f"Loaded {len(text)} characters from PDF files.")


Loaded 128973 characters from PDF files.


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_text(text)

print(f"Split into {len(chunks)} chunks.")

Split into 289 chunks.


In [3]:
from src.basic_code_search.embedding_model import EmbeddingModel

embedding_model = EmbeddingModel()
embeddings = embedding_model.encode(chunks)

print(f"Generated {len(embeddings)} embeddings.")

Generated 289 embeddings.


In [4]:
from src.basic_code_search.search_engine import SearchEngine

search_engine = SearchEngine(embedding_model=embedding_model, collection_name="basic_search_db", top_k=3)
search_engine.open()
search_engine.load_text_data(chunks, batch_size=128)
query = "What is a transformer model?"
results = search_engine.search(query)
for result in results:
    print(f"\nResult {result.id} (Score: {result.score}):\n{result.payload['text']}")
search_engine.close()


Result 1df92e94-2239-45d3-a206-167c917edb3b (Score: 0.5118749):
language modeling tasks [34].
To the best of our knowledge, however, the Transformer is the first transduction model relying
entirely on self-attention to compute representations of its input and output without using sequence-
aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate
self-attention and discuss its advantages over models such as [17, 18] and [9].
3 Model Architecture

Result b41b4495-857e-4f6a-9707-cc5d756378b5 (Score: 0.47969738):
block, computing hidden representations in parallel for all input and output positions. In these models,
the number of operations required to relate signals from two arbitrary input or output positions grows
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is

Result c7be0e96-8a36-4680-bffe-f4

# Task 2

In [5]:
from datasets import load_dataset

corpus = load_dataset("CoIR-Retrieval/cosqa", "corpus")['corpus']
queries = load_dataset("CoIR-Retrieval/cosqa", "queries")['queries']
default = load_dataset("CoIR-Retrieval/cosqa", "default")

In [6]:
print(default)

DatasetDict({
    train: Dataset({
        features: ['query-id', 'corpus-id', 'score'],
        num_rows: 19604
    })
    test: Dataset({
        features: ['query-id', 'corpus-id', 'score'],
        num_rows: 500
    })
    valid: Dataset({
        features: ['query-id', 'corpus-id', 'score'],
        num_rows: 500
    })
})


In [7]:
print(queries[0])

{'_id': 'q1', 'partition': 'train', 'text': 'python code to write bool value 1', 'title': '', 'language': '', 'meta_information': {'dummy_field': ''}}


In [8]:
print(corpus[0])

{'_id': 'd1', 'partition': 'train', 'text': 'def writeBoolean(self, n):\n        """\n        Writes a Boolean to the stream.\n        """\n        t = TYPE_BOOL_TRUE\n\n        if n is False:\n            t = TYPE_BOOL_FALSE\n\n        self.stream.write(t)', 'title': '', 'language': 'PYTHON', 'meta_information': {'dummy_field': ''}}


In [9]:
pd_queries = queries.to_pandas()
pd_corpus = corpus.to_pandas()
pd_test = default['test'].to_pandas()

test_full_texts = pd_test.merge(pd_queries, left_on='query-id', right_on='_id', how='left')[['query-id', 'corpus-id', 'text', 'score']].rename(columns={'text': 'query_text'})
test_full_texts = test_full_texts.merge(pd_corpus, left_on='corpus-id', right_on='_id', how='left')[['query-id', 'corpus-id', 'query_text', 'text', 'score']].rename(columns={'text': 'corpus_text'})

print(test_full_texts.head())

  query-id corpus-id                             query_text  \
0   q20105    d20105       sort by a token in string python   
1   q20106    d20106          python check file is readonly   
2   q20107    d20107  declaring empty numpy array in python   
3   q20108    d20108  test for iterable is string in python   
4   q20109    d20109     python print results of query loop   

                                         corpus_text  score  
0  def _process_and_sort(s, force_ascii, full_pro...      1  
1  def is_readable(filename):\n    """Check if fi...      1  
2  def empty(self, name, **kwargs):\n        """C...      1  
3  def is_iterable_but_not_string(obj):\n    """\...      1  
4  def print_runs(query):\n    """ Print all rows...      1  


In [10]:
from src.basic_code_search.search_engine import SearchEngine # DEBUG, remove later
from src.basic_code_search.embedding_model import EmbeddingModel # DEBUG, remove later
import traceback

embedding_model = EmbeddingModel() # DEBUG, remove later

# start search engine
search_engine = SearchEngine(embedding_model=embedding_model, collection_name="basic_search_db", top_k=10)
try:
    # open connection to vector database
    search_engine.open()

    # load corpus into search engine
    search_engine.load_text_data(ids=list(corpus['_id']), texts=list(corpus['text']), batch_size=2048, verbose=True)

    model_responses = []
    for test_query in test_full_texts['query_text']:
        results = search_engine.search(test_query)
        model_responses.append([res.payload['dataset_id'] for res in results])

    print() # empty line for better readability
    search_engine.evaluate(predictions=model_responses, targets=list(test_full_texts['corpus-id']))

    search_engine.close()
except Exception as e:
    search_engine.close()
    print(traceback.print_exc())

Encoding 20604 texts into embeddings...
Encoding done.

Upserting 20604 vectors into the database in batches of 2048...
Upserted batch 1 with 2048 vectors.
Upserted batch 2 with 2048 vectors.
Upserted batch 3 with 2048 vectors.
Upserted batch 4 with 2048 vectors.
Upserted batch 5 with 2048 vectors.
Upserted batch 6 with 2048 vectors.
Upserted batch 7 with 2048 vectors.
Upserted batch 8 with 2048 vectors.
Upserted batch 9 with 2048 vectors.
Upserted batch 10 with 2048 vectors.
Upserted batch 11 with 124 vectors.

Evaluation Metrics:
Recall@10: 0.4800
MRR@10: 0.2383
NDCG@10: 0.2959


# Task 3

In [11]:
from datasets import Dataset

# prepare train and eval datasets
datasets = {
    'train': default['train'],
    'valid': default['valid'],
}

for name, dataset in datasets.items():
    current_ds = dataset.to_pandas()
    current_ds = current_ds[['query-id', 'corpus-id', 'score']]

    # create new column with negative relation
    last_corpus_id = current_ds['corpus-id'].iloc[-1]
    current_ds['negative_corpus-id'] = current_ds['corpus-id'].shift(1, fill_value=last_corpus_id)

    # leave only positive samples
    current_ds = current_ds[current_ds['score'] == 1]
    current_ds = current_ds[['query-id', 'corpus-id', 'negative_corpus-id']]

    # get full texts from queries and corpus
    current_ds = current_ds.merge(pd_queries, left_on='query-id', right_on='_id', how='left')[['query-id', 'corpus-id', 'negative_corpus-id', 'text']].rename(columns={'text': 'anchor'})
    current_ds = current_ds.merge(pd_corpus, left_on='corpus-id', right_on='_id', how='left')[['anchor', 'text', 'negative_corpus-id']].rename(columns={'text': 'positive'})
    current_ds = current_ds.merge(pd_corpus, left_on='negative_corpus-id', right_on='_id', how='left')[['anchor', 'positive', 'text']].rename(columns={'text': 'negative'})

    # shuffle dataset
    current_ds = current_ds.sample(frac=1, random_state=42).reset_index(drop=True)
    # save as HuggingFace Dataset
    current_ds = current_ds.to_dict(orient='list')
    current_ds = Dataset.from_dict(current_ds)

    # update datasets dictionary
    datasets[name] = current_ds
    
    print(name, current_ds)

train Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 9020
})
valid Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 500
})


In [12]:
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers import SentenceTransformerTrainingArguments

fine_tuned_model = EmbeddingModel()

args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir="results/fine-tuned/cosqa-mnr-loss",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=100,
    run_name="basic-search-engine-cosqa-mnr",  # Will be used in W&B if `wandb` is installed
)

fine_tuned_model.train(
    train_dataset=datasets['train'],
    eval_dataset=datasets['valid'],
    loss_function=MultipleNegativesRankingLoss,
    args=args
)   

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss
100,0.180800,0.130687
200,0.143700,0.120005
300,0.158900,0.120613
400,0.137500,0.114525
500,0.141500,0.112438


In [13]:
search_engine = SearchEngine(embedding_model=fine_tuned_model, collection_name="basic_search_db", top_k=10)

try:
    # open connection to vector database
    search_engine.open()

    # load corpus into search engine
    search_engine.load_text_data(ids=list(corpus['_id']), texts=list(corpus['text']), batch_size=2048, verbose=True)

    model_responses = []
    for test_query in test_full_texts['query_text']:
        results = search_engine.search(test_query)
        model_responses.append([res.payload['dataset_id'] for res in results])

    print() # empty line for better readability
    search_engine.evaluate(predictions=model_responses, targets=list(test_full_texts['corpus-id']))

    search_engine.close()
except Exception as e:
    search_engine.close()
    print(traceback.print_exc())

Encoding 20604 texts into embeddings...
Encoding done.

Upserting 20604 vectors into the database in batches of 2048...
Upserted batch 1 with 2048 vectors.
Upserted batch 2 with 2048 vectors.
Upserted batch 3 with 2048 vectors.
Upserted batch 4 with 2048 vectors.
Upserted batch 5 with 2048 vectors.
Upserted batch 6 with 2048 vectors.
Upserted batch 7 with 2048 vectors.
Upserted batch 8 with 2048 vectors.
Upserted batch 9 with 2048 vectors.
Upserted batch 10 with 2048 vectors.
Upserted batch 11 with 124 vectors.

Evaluation Metrics:
Recall@10: 0.5340
MRR@10: 0.2475
NDCG@10: 0.3161


In [14]:
from datasets import Dataset

# prepare train and eval datasets
datasets = {
    'train': default['train'],
    'valid': default['valid'],
}

for name, dataset in datasets.items():
    current_ds = dataset.to_pandas()

    # get full texts from queries and corpus
    current_ds = current_ds.merge(pd_queries, left_on='query-id', right_on='_id', how='left')[['query-id', 'corpus-id', 'text', 'score']].rename(columns={'text': 'anchor'})
    current_ds = current_ds.merge(pd_corpus, left_on='corpus-id', right_on='_id', how='left')[['anchor', 'text', 'score']].rename(columns={'text': 'positive/negative', 'score': 'label'})

    # shuffle dataset
    current_ds = current_ds.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # save as HuggingFace Dataset
    current_ds = current_ds.to_dict(orient='list')
    current_ds = Dataset.from_dict(current_ds)

    # update datasets dictionary
    datasets[name] = current_ds
    
    print(name, current_ds)

train Dataset({
    features: ['anchor', 'positive/negative', 'label'],
    num_rows: 19604
})
valid Dataset({
    features: ['anchor', 'positive/negative', 'label'],
    num_rows: 500
})


In [15]:
from sentence_transformers.losses import ContrastiveLoss

fine_tuned_model = EmbeddingModel()

args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir="results/fine-tuned/cosqa-contrastive-loss",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=100,
    run_name="basic-search-engine-cosqa-contrastive",  # Will be used in W&B if `wandb` is installed
)

fine_tuned_model.train(
    train_dataset=datasets['train'],
    eval_dataset=datasets['valid'],
    loss_function=ContrastiveLoss,
    args=args # keep arguments consistent
)   

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss
100,0.035900,0.037284
200,0.032200,0.039816
300,0.032300,0.033437
400,0.031400,0.030155
500,0.031400,0.031368
600,0.031700,0.029900
700,0.031500,0.036921
800,0.030800,0.030629
900,0.031200,0.033813
1000,0.031900,0.034316


In [16]:
search_engine = SearchEngine(embedding_model=fine_tuned_model, collection_name="basic_search_db", top_k=10)

try:
    # open connection to vector database
    search_engine.open()

    # load corpus into search engine
    search_engine.load_text_data(ids=list(corpus['_id']), texts=list(corpus['text']), batch_size=2048, verbose=True)

    model_responses = []
    for test_query in test_full_texts['query_text']:
        results = search_engine.search(test_query)
        model_responses.append([res.payload['dataset_id'] for res in results])

    print() # empty line for better readability
    search_engine.evaluate(predictions=model_responses, targets=list(test_full_texts['corpus-id']))

    search_engine.close()
except Exception as e:
    search_engine.close()
    print(traceback.print_exc())

Encoding 20604 texts into embeddings...
Encoding done.

Upserting 20604 vectors into the database in batches of 2048...
Upserted batch 1 with 2048 vectors.
Upserted batch 2 with 2048 vectors.
Upserted batch 3 with 2048 vectors.
Upserted batch 4 with 2048 vectors.
Upserted batch 5 with 2048 vectors.
Upserted batch 6 with 2048 vectors.
Upserted batch 7 with 2048 vectors.
Upserted batch 8 with 2048 vectors.
Upserted batch 9 with 2048 vectors.
Upserted batch 10 with 2048 vectors.
Upserted batch 11 with 124 vectors.

Evaluation Metrics:
Recall@10: 0.4280
MRR@10: 0.1960
NDCG@10: 0.2513
